<a href="https://colab.research.google.com/github/Ayush2005292/ML/blob/main/yolov8_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [26]:

!unzip "/content/data.zip" -d "/content/"

unzip:  cannot find or open /content/data.zip, /content/data.zip.zip or /content/data.zip.ZIP.


### 4. Create `data.yaml` file

This file defines the paths to your dataset and the class names for object detection. You'll need to customize `train`, `val`, `test` paths, and the `names` list according to your dataset structure and classes. For now, I'll create a placeholder. If your dataset is structured differently, please update these paths accordingly.

In [27]:
import os

# Define the content for data.yaml
data_yaml_content = """
path: /content  # Dataset root directory
train: train/images  # Train images relative to path
val: val/images    # Val images relative to path
test: test/images    # Test images relative to path

# Classes
nc: 2  # number of classes
names: ['class1', 'class2']  # class names
"""

# Write the content to data.yaml
with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("data.yaml created successfully at /content/data.yaml")
print("Please review and adjust the paths and class names in /content/data.yaml to match your dataset.")

data.yaml created successfully at /content/data.yaml
Please review and adjust the paths and class names in /content/data.yaml to match your dataset.


In [4]:
### 3. Install packages ###

!git clone https://github.com/autogyro/yolo-V8.git
!cd yolo-V8/ && pip install ultralytics

Cloning into 'yolo-V8'...
remote: Enumerating objects: 2723, done.
remote: Total 2723 (delta 0), reused 0 (delta 0), pack-reused 2723 (from 1)
Receiving objects: 100% (2723/2723), 1.41 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (1854/1854), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.3 MB/s eta 0:00:00


In [16]:
### 4. Train model ###

import os

from ultralytics import YOLO


config_path = '/content/data.yaml'


!yolo task=detect mode=train model=yolov8n.pt data={config_path} epochs=50 imgsz=640 plots=True

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

In [17]:
# ------------------ Fix for Encoding Issues ------------------
# Sometimes, especially on platforms like Google Colab or Windows,
# you might face locale/encoding issues when running external commands.
# So, you override 'getpreferredencoding' to always return 'UTF-8'.

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"

# Patch the 'locale' module's getpreferredencoding method
locale.getpreferredencoding = getpreferredencoding

# ------------------ Run YOLOv8 Validation ------------------
# This command runs validation (evaluation) of a trained YOLO model.

!yolo task=detect mode=val model=runs/detect/train/weights/best.pt data=/content/data.yaml save_json=True

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [18]:
from IPython.display import Image, display
import glob

!yolo predict model=/content/runs/detect/train/weights/best.pt \
              source=/content/test/images/y701_jpg.rf.81a4472f77fdc1f31537342ceca340c9.jpg \
              project=/content/runs/detect \
              name=predict \
              exist_ok=True


# Find the path of the predicted image (YOLOv8 saves output in 'runs/detect/predict' by default)
predicted_images = glob.glob('/content/runs/detect/predict/*.jpg')  # or .png if needed

# Display the first predicted image
if predicted_images:
    display(Image(filename=predicted_images[0]))
else:
    print("No predicted images found.")


Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [20]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Paths
test_folder = '/content/test/images'
output_folder = '/content/runs/detect'
output_name = 'predictions'
output_path = os.path.join(output_folder, output_name)

# Ensure output directory exists
os.makedirs(output_path, exist_ok=True)

# Load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Select up to 16 test images
test_images = [os.path.join(test_folder, img) for img in os.listdir(test_folder) if img.endswith(('.jpg', '.png'))][:16]

# Run predictions and save
results = []
for img_path in test_images:
    result = model.predict(source=img_path, save=True, save_txt=True, project=output_folder, name=output_name, exist_ok=True)
    results.append(result)

# Gather predicted image paths
predicted_images = [os.path.join(output_path, os.path.basename(img_path)) for img_path in test_images]

# Display images in 4x4 grid
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, img_path in enumerate(predicted_images):
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(img_path), fontsize=8)
        axes[i].axis('off')
    else:
        axes[i].axis('off')
        axes[i].set_title("Image not found", fontsize=8)

plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/train/weights/best.pt'

In [24]:
!ls -F /content

data.yaml  gdrive/  runs/  runs.zip  sample_data/  yolo-V8/  yolov8n.pt


In [22]:
import os

file_path = '/content/runs/detect/train/weights/best.pt'

if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does not exist.")

The file '/content/runs/detect/train/weights/best.pt' does not exist.


In [23]:
### 5. Download results ###

from google.colab import files


!zip -r /content/runs.zip /content/runs

files.download('/content/runs.zip')

updating: content/runs/ (stored 0%)
updating: content/runs/detect/ (stored 0%)
updating: content/runs/detect/train/ (stored 0%)
updating: content/runs/detect/train/weights/ (stored 0%)
updating: content/runs/detect/train/args.yaml (deflated 53%)
updating: content/runs/detect/predictions/ (stored 0%)
  adding: content/runs/detect/train-3/ (stored 0%)
  adding: content/runs/detect/train-3/weights/ (stored 0%)
  adding: content/runs/detect/train-3/args.yaml (deflated 53%)
  adding: content/runs/detect/train-2/ (stored 0%)
  adding: content/runs/detect/train-2/weights/ (stored 0%)
  adding: content/runs/detect/train-2/args.yaml (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>